# RQ2: MLP

**Research Question:** Does adding ACE and/or SDHE features improve prediction of frequent mental distress beyond traditional features?

**4 Feature Configurations:**
- **(A) Baseline:** Traditional features only (same as RQ1)
- **(B) Baseline + ACE:** + 13 ACE items + ACE_score
- **(C) Baseline + SDHE:** + 10 SDHE items + SDHE_burden
- **(D) Baseline + ACE + SDHE + interaction:** All features

In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported")

## Mount on Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Load dataset

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/COMP90049/processed"

df_rq2 = pd.read_csv(f"{SAVE_DIR}/df_rq2_augmentation.csv")

print(f"RQ2 dataset: {df_rq2.shape}")
print(f"\nTarget distribution:")
print(df_rq2['target_fmd'].value_counts())
print(df_rq2['target_fmd'].value_counts(normalize=True))
print(f"\nColumns ({len(df_rq2.columns)}):")
print(df_rq2.columns.tolist())

## Define feature configurations

In [ ]:
# Config A: Baseline
baseline_features = [
    # demographic
    '_AGE_G', '_RACE1', 'EDUCA', 'INCOME3', 'MARITAL', 'EMPLOY1', 'CHILDREN',
    # behavioral
    'EXERANY2', 'SLEPTIM1', 'SMOKE100', 'alcohol_days_30', 'BMI',
    # chronic
    'DIABETE4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3',
    'CHCCOPD3', 'HAVARTH4', 'CHCKDNY2', 'ADDEPEV3',
    'chronic_count',
    # healthcare access
    'PRIMINSR', 'PERSDOC3', 'MEDCOST1', 'CHECKUP1'
]

# Config B: baseline + ACE
ace_features = [
    'ACEDEPRS', 'ACEDRINK', 'ACEDRUGS', 'ACEPRISN', 'ACEDIVRC',
    'ACEPUNCH', 'ACEHURT1', 'ACESWEAR', 'ACETOUCH', 'ACETTHEM',
    'ACEHVSEX', 'ACEADSAF', 'ACEADNED',
    'ACE_score'
]

# Config C: baseline + SDHE
sdhe_features = [
    'LSATISFY', 'EMTSUPRT', 'SDHISOLT', 'SDHEMPLY', 'FOODSTMP',
    'SDHFOOD1', 'SDHBILLS', 'SDHUTILS', 'SDHTRNSP', 'SDHSTRE1',
    'SDHE_burden'
]

# Config D: baseline + ACE + SDHE + interaction
interaction_features = ['ACE_x_SDHE']

configs = {
    'A_baseline': [c for c in baseline_features if c in df_rq2.columns],
    'B_ace':      [c for c in baseline_features + ace_features if c in df_rq2.columns],
    'C_sdhe':     [c for c in baseline_features + sdhe_features if c in df_rq2.columns],
    'D_all':      [c for c in baseline_features + ace_features + sdhe_features + interaction_features if c in df_rq2.columns]
}

# Remove duplicates while preserving order
for key in configs:
    configs[key] = list(dict.fromkeys(configs[key]))

print("Feature configurations:")
for name, cols in configs.items():
    print(f'  {name}: {len(cols)} features')
print(f"\nNew features in B vs A: {set(configs['B_ace']) - set(configs['A_baseline'])}")
print(f"New features in C vs A: {set(configs['C_sdhe']) - set(configs['A_baseline'])}")
print(f"New features in D vs A: {set(configs['D_all']) - set(configs['A_baseline'])}")

## Define feature groups

In [ ]:
# Numeric features (continuous -- get StandardScaler)
all_numeric = {
    'CHILDREN', 'SLEPTIM1', 'alcohol_days_30', 'BMI', 'chronic_count',
    'ACE_score', 'SDHE_burden', 'ACE_x_SDHE'
}

# INCOME3 treated as ordinal-missing: impute with -1, then one-hot
income_col = 'INCOME3'

def get_feature_groups(feature_list):
    numeric     = [c for c in feature_list if c in all_numeric]
    income      = [income_col] if income_col in feature_list else []
    categorical = [c for c in feature_list if c not in all_numeric and c != income_col]
    return numeric, categorical, income

for name, cols in configs.items():
    num, cat, inc = get_feature_groups(cols)
    print(f"{name}: {len(num)} numeric, {len(cat)} categorical, {len(inc)} income")

## Train / Validation / Test split

In [ ]:
X_all = df_rq2.drop(columns=['target_fmd'])
y_all = df_rq2['target_fmd']

# 80% train+val  /  20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

# 75% of train_val -> train (=60% total) / 25% -> val (=20% total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)

print(f"Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}")
print(f"Train pos rate: {y_train.mean():.3f}")
print(f"Val pos rate:   {y_val.mean():.3f}")
print(f"Test pos rate:  {y_test.mean():.3f}")

## Preprocessing function

In [ ]:
def build_preprocessor(feature_list):
    numeric, categorical, income = get_feature_groups(feature_list)
    transformers = []

    if numeric:
        transformers.append(('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler',  StandardScaler())
        ]), numeric))

    if categorical:
        transformers.append(('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot',  OneHotEncoder(handle_unknown='ignore'))
        ]), categorical))

    if income:
        transformers.append(('income', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value=-1)),
            ('onehot',  OneHotEncoder(handle_unknown='ignore'))
        ]), income))

    return ColumnTransformer(transformers=transformers)

## Hyperparameter tuning function

Same grid as RQ1 MLP:
- `hidden_layer_sizes`: (64,), (128,), (64, 32), (128, 64), (128, 64, 32)
- `learning_rate_init`: 0.0001, 0.001, 0.01
- `alpha` (L2 regularisation): 0.0001, 0.001, 0.01

Selection criterion: **balanced accuracy** on validation set (same as RQ1).

> Note: `MLPClassifier.fit()` does not accept `sample_weight`. Class imbalance is instead handled by pre-computing `compute_sample_weight('balanced', y_train)` and passing it during `.fit()`.

In [ ]:
def tune_mlp(config_name, feature_list, X_train, y_train, X_val, y_val):
    """Grid-search MLP hyperparameters; return best params Series and full results DataFrame."""

    # Pre-process once for this config to avoid redundant fitting across all combos
    preprocessor = build_preprocessor(feature_list)
    X_tr = preprocessor.fit_transform(X_train[feature_list], y_train)
    X_vl = preprocessor.transform(X_val[feature_list])

    # MLPClassifier has no class_weight param; use sample_weight instead
    sw_train = compute_sample_weight('balanced', y_train)

    # Hyperparameter grid (mirrors RQ1 MLP)
    hidden_layer_candidates = [
        (64,),
        (128,),
        (64, 32),
        (128, 64),
        (128, 64, 32),
    ]
    lr_candidates    = [0.0001, 0.001, 0.01]
    alpha_candidates = [0.0001, 0.001, 0.01]   # L2 regularisation strength

    total = len(hidden_layer_candidates) * len(lr_candidates) * len(alpha_candidates)
    print(f"\n{'='*70}")
    print(f"Tuning {total} MLP configs for {config_name}")
    print(f"{'='*70}")

    results = []
    start   = time.time()
    i       = 0

    for hidden_layers in hidden_layer_candidates:
        for lr in lr_candidates:
            for alpha in alpha_candidates:
                i += 1
                clf = MLPClassifier(
                    hidden_layer_sizes=hidden_layers,
                    activation='relu',
                    learning_rate_init=lr,
                    alpha=alpha,
                    max_iter=500,
                    early_stopping=True,
                    n_iter_no_change=30,
                    validation_fraction=0.1,
                    random_state=42,
                )
                clf.fit(X_tr, y_train, sample_weight=sw_train)

                y_pred = clf.predict(X_vl)
                y_prob = clf.predict_proba(X_vl)[:, 1]

                results.append({
                    'hidden_layers':    str(hidden_layers),
                    'lr':               lr,
                    'alpha':            alpha,
                    'n_iter':           clf.n_iter_,
                    'val_balanced_acc': balanced_accuracy_score(y_val, y_pred),
                    'val_f1':           f1_score(y_val, y_pred, zero_division=0),
                    'val_roc_auc':      roc_auc_score(y_val, y_prob),
                    'val_pr_auc':       average_precision_score(y_val, y_prob),
                    'val_recall':       recall_score(y_val, y_pred, zero_division=0),
                    'val_precision':    precision_score(y_val, y_pred, zero_division=0),
                })

                if i % 9 == 0 or i == total:
                    elapsed = time.time() - start
                    eta     = (total - i) * (elapsed / i)
                    best_so_far = max(r['val_balanced_acc'] for r in results)
                    print(f"  [{i}/{total}] Best bal_acc={best_so_far:.4f} | ETA {eta/60:.1f}min")

    results_df = pd.DataFrame(results).sort_values('val_balanced_acc', ascending=False)
    elapsed = time.time() - start
    print(f"Done in {elapsed/60:.1f} min | Best val balanced_acc = {results_df.iloc[0]['val_balanced_acc']:.4f}")

    return results_df.iloc[0], results_df

## Evaluation function

In [ ]:
def eval_on_test(config_name, feature_list, best_params, X_train_val, y_train_val, X_test, y_test):
    """Retrain on full train+val with best params, then evaluate on held-out test set."""
    import ast

    # Parse best params back to correct Python types
    hidden_layers = ast.literal_eval(best_params['hidden_layers'])
    lr            = float(best_params['lr'])
    alpha         = float(best_params['alpha'])

    # Preprocess on full train_val split
    preprocessor = build_preprocessor(feature_list)
    X_tv = preprocessor.fit_transform(X_train_val[feature_list], y_train_val)
    X_te = preprocessor.transform(X_test[feature_list])

    sw_tv = compute_sample_weight('balanced', y_train_val)

    clf = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation='relu',
        learning_rate_init=lr,
        alpha=alpha,
        max_iter=500,
        early_stopping=True,
        n_iter_no_change=30,
        validation_fraction=0.1,
        random_state=42,
    )
    clf.fit(X_tv, y_train_val, sample_weight=sw_tv)

    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]

    metrics = {
        'Config':        config_name,
        'Features':      len(feature_list),
        'hidden_layers': str(hidden_layers),
        'lr':            lr,
        'alpha':         alpha,
        'Accuracy':      accuracy_score(y_test, y_pred),
        'Balanced_Acc':  balanced_accuracy_score(y_test, y_pred),
        'Precision':     precision_score(y_test, y_pred, zero_division=0),
        'Recall':        recall_score(y_test, y_pred, zero_division=0),
        'F1':            f1_score(y_test, y_pred, zero_division=0),
        'ROC_AUC':       roc_auc_score(y_test, y_prob),
        'PR_AUC':        average_precision_score(y_test, y_prob),
    }

    print(f"\n--- {config_name} Test Results ---")
    print(f"  hidden={hidden_layers}  lr={lr}  alpha={alpha}")
    print(f"  F1={metrics['F1']:.4f}  AUC={metrics['ROC_AUC']:.4f}  "
          f"Recall={metrics['Recall']:.4f}  Precision={metrics['Precision']:.4f}")

    return metrics, clf, preprocessor, y_pred, y_prob

## Run all 4 configurations

In [ ]:
all_best_params    = {}
all_tuning_results = {}
all_test_metrics   = []
all_models         = {}   # stores (clf, preprocessor) tuples
all_preds          = {}
all_probs          = {}

for config_name, feature_list in configs.items():
    best_params, tuning_df = tune_mlp(
        config_name, feature_list, X_train, y_train, X_val, y_val)
    all_best_params[config_name]    = best_params
    all_tuning_results[config_name] = tuning_df

    metrics, clf, preprocessor, y_pred, y_prob = eval_on_test(
        config_name, feature_list, best_params,
        X_train_val, y_train_val, X_test, y_test)

    all_test_metrics.append(metrics)
    all_models[config_name] = (clf, preprocessor)
    all_preds[config_name]  = y_pred
    all_probs[config_name]  = y_prob

print("\n" + "="*70)
print("All 4 configurations completed!")
print("="*70)

## Results summary

In [ ]:
results_table = pd.DataFrame(all_test_metrics)

baseline_f1  = results_table.loc[results_table['Config'] == 'A_baseline', 'F1'].values[0]
baseline_auc = results_table.loc[results_table['Config'] == 'A_baseline', 'ROC_AUC'].values[0]

results_table['\u0394F1']  = results_table['F1']     - baseline_f1
results_table['\u0394AUC'] = results_table['ROC_AUC'] - baseline_auc

results_table['\u0394F1_str']  = results_table['\u0394F1'].apply(
    lambda x: f"+{x:.4f}" if x > 0 else (f"{x:.4f}" if x != 0 else "-"))
results_table['\u0394AUC_str'] = results_table['\u0394AUC'].apply(
    lambda x: f"{x:.4f}" if x > 0 else (f"{x:.4f}" if x != 0 else "-"))

print("="*60)
print("RQ2 Results: MLP -- Feature Augmentation Comparison")
print("="*60)
display(results_table[[
    'Config', 'Features', 'hidden_layers', 'lr', 'alpha',
    'F1', '\u0394F1_str', 'ROC_AUC', '\u0394AUC_str',
    'Recall', 'Precision', 'Balanced_Acc', 'PR_AUC'
]].round(4))

b_delta = results_table.loc[results_table['Config'] == 'B_ace',  '\u0394F1'].values[0]
c_delta = results_table.loc[results_table['Config'] == 'C_sdhe', '\u0394F1'].values[0]
d_delta = results_table.loc[results_table['Config'] == 'D_all',  '\u0394F1'].values[0]

print(f"\n\u0394F1 (B vs A):  {b_delta:.4f}  <- ACE alone")
print(f"\u0394F1 (C vs A):  {c_delta:.4f}  <- SDHE alone")
print(f"\u0394F1 (D vs A):  {d_delta:.4f}  <- ACE + SDHE + interaction")

# Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

config_labels = ['A: Baseline', 'B: +ACE', 'C: +SDHE', 'D: +All']
colors = ['#95a5a6', '#9b59b6', '#e67e22', '#27ae60']

# Plot 1: F1 comparison with delta labels
f1_vals = results_table['F1'].values
bars = axes[0].bar(config_labels, f1_vals, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('F1 Score by Feature Configuration', fontweight='bold')
axes[0].set_ylabel('F1 Score')
for i, (bar, val) in enumerate(zip(bars, f1_vals)):
    delta = val - f1_vals[0]
    label = f'{val:.3f}' if i == 0 else f'{val:.3f}\n(\u0394{delta:+.3f})'
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                 label, ha='center', va='bottom', fontsize=9)
axes[0].set_ylim(0, max(f1_vals) * 1.18)

# Plot 2: ROC-AUC comparison
auc_vals = results_table['ROC_AUC'].values
bars = axes[1].bar(config_labels, auc_vals, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('ROC-AUC by Feature Configuration', fontweight='bold')
axes[1].set_ylabel('ROC-AUC')
for i, (bar, val) in enumerate(zip(bars, auc_vals)):
    delta = val - auc_vals[0]
    label = f'{val:.3f}' if i == 0 else f'{val:.3f}\n(\u0394{delta:+.3f})'
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                 label, ha='center', va='bottom', fontsize=9)
axes[1].set_ylim(0, max(auc_vals) * 1.15)

# Plot 3: ROC curves overlaid
for name, color, label in zip(configs.keys(), colors, config_labels):
    RocCurveDisplay.from_predictions(
        y_test, all_probs[name], name=label,
        ax=axes[2], plot_chance_level=False, color=color)
axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[2].set_title('ROC Curves: All Configurations', fontweight='bold')
axes[2].legend(fontsize=8, loc='lower right')

plt.suptitle('RQ2: MLP -- Feature Augmentation Comparison', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/rq2_mlp_config_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: rq2_mlp_config_comparison.png")

## Classification reports & confusion matrices

In [ ]:
for name in configs.keys():
    print(f"\n{'='*60}")
    print(f"Config {name} -- Classification Report")
    print(f"{'='*60}")
    print(classification_report(y_test, all_preds[name]))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, all_preds[name]))

## Best hyperparameters per configuration

In [ ]:
print("Best hyperparameters selected per configuration (by val balanced accuracy):\n")
for name, params in all_best_params.items():
    print(f"  {name}:")
    print(f"    hidden_layers    = {params['hidden_layers']}")
    print(f"    lr               = {params['lr']}")
    print(f"    alpha            = {params['alpha']}")
    print(f"    val_balanced_acc = {params['val_balanced_acc']:.4f}")
    print()

## Save results

In [ ]:
results_table.to_csv(f'{SAVE_DIR}/rq2_mlp_comparison.csv', index=False)

for name, df in all_tuning_results.items():
    df.to_csv(f'{SAVE_DIR}/rq2_mlp_tuning_{name}.csv', index=False)

# Final summary
print("="*70)
print("RQ2 MLP -- FINAL SUMMARY")
print("="*70)

for _, row in results_table.iterrows():
    print(f"\n  Config {row['Config']}:")
    print(f"    Features:      {int(row['Features'])}")
    print(f"    hidden_layers: {row['hidden_layers']}")
    print(f"    lr={row['lr']}  alpha={row['alpha']}")
    print(f"    F1={row['F1']:.4f}  AUC={row['ROC_AUC']:.4f}  \u0394F1={row['\u0394F1']:+.4f}")

best_config = results_table.loc[results_table['F1'].idxmax()]
print(f"\nBest config: {best_config['Config']} (F1={best_config['F1']:.4f})")
print("\nAll results saved.")